# Query-gen check — QUERY_GEN_SPEC.md v3, end to end

Tests `agent/query_gen.py` (the QUERY_GEN_SPEC.md v3 track — "Two
mechanisms: route the question to a document, then close the citation
graph. Nothing else.") against the full 1071-chunk corpus (QCVN stays in
under v3 — routing keeps it from being a distractor without dropping the
133 items whose only gold is QCVN). Separate from
`notebooks/phase1_retrieval_check.ipynb` (the shipped v2.8
`agent/pipeline.py` track, still QCVN-blind) and
`notebooks/manual_retrieval_check.ipynb` (raw hand-written queries, no
routing/graph).

**No live LLM call is needed for the query text under v3** — unlike the
superseded v2 draft's channel-tagged multi-query FORMULATE, v3 routes
using the question + `perception.observed`, and retrieves using whatever
`perception.violation` (or a hand-written stand-in) already is. So Step 2
below is where you write ONE `MY_PERCEPTION` (focus/vehicle/observed/
violation) by hand, standing in for a real PERCEIVE call.

Workflow:
1. Pick an item — see its question, options, gold chunk ids.
2. Write `MY_PERCEPTION` by hand (stand-in for PERCEIVE's output).
3. See the routing decision (§1) — which document(s) it selects, and why.
4. Retrieve within the routed documents (§2) — before any graph expansion.
5. Expand via the citation graph (§3) — see what gets added, and tagged
   `graph_rev`/`graph_fwd`, on top of the base retrieval.
6. Retrieved (all sources) vs. gold, side by side.

**Kernel**: needs `agent`'s retrieval stack (pyserini/pyvi for `--retriever
bm25`) but no LLM/GPU — `agent/query_gen.py` never loads a model.

**Before running with `RETRIEVER = "bm25"`**: the BM25 index must be built
against the CURRENT `law_db.json` (1071 chunks, QCVN restored under v3) —
rebuild it with `python -m agent.build_bm25_index` if your index still
reflects an earlier corpus version. Use `RETRIEVER = "mock"` to skip that
dependency entirely.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agent").exists():
    ROOT = ROOT.parent  # running from notebooks/, repo root is one level up
assert (ROOT / "agent").exists(), f"can't find the agent/ package from {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agent.corpus import Corpus
from agent.resolve import CitationIndex
from agent.retrieval.bm25 import DEFAULT_INDEX_DIR, BM25Retriever
from agent.retrieval.mock import MockRetriever
from agent.schemas import Perception
from agent.query_gen import (
    ROUTE_LABEL_TO_DOC,
    expand_with_graph,
    load_citation_graph,
    retrieve_within_routed,
    route,
)

## Config — edit this cell, then run everything below

In [ ]:
# --- data ---
DATA_PATH = ROOT / "law.json"
CORPUS_PATH = ROOT / "law_db.json"
GRAPH_PATH = ROOT / "banks" / "citation_graph.json"

# --- retriever: "bm25" (real, needs the Lucene index built against the
#     CURRENT law_db.json, QCVN included) or "mock" (deterministic, no index) ---
RETRIEVER = "bm25"
INDEX_DIR = DEFAULT_INDEX_DIR
BM25_K1, BM25_B = 0.9, 0.4

K_TOTAL = 12     # total per-round retrieval budget, split evenly across routed docs (§2)
MAX_REV = 8      # citation-graph reverse-expansion cap (§3)
MAX_FWD = 8      # citation-graph forward-expansion cap, after BM25 rerank (§3)
AS_OF = None     # effective_date filter, e.g. "2025-01-01"

ITEM_INDEX = 0   # index into `records`

In [ ]:
records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
corpus = Corpus.load(CORPUS_PATH)
citation_index = CitationIndex.from_file(CORPUS_PATH)
graph = load_citation_graph(GRAPH_PATH)

if RETRIEVER == "bm25":
    retriever = BM25Retriever(index_dir=INDEX_DIR, k1=BM25_K1, b=BM25_B)
else:
    retriever = MockRetriever(corpus)

print(f"{len(records)} questions | corpus: {len(corpus)} chunks (v{corpus.version}) | retriever={RETRIEVER}")
print(f"citation graph: {len(graph['forward'])} forward, {len(graph['reverse'])} reverse entries")

## Step 1 — pick an item, see its question, options, and gold

In [ ]:
rec = records[ITEM_INDEX]
gold_ids = citation_index.resolve_set(rec.get("law", []))

print(f"item_id={rec.get('id')}  type_image={rec.get('type_image')}")
print(f"Q: {rec['question']}\n")
for L in ("A", "B", "C", "D"):
    marker = " <- GT" if rec.get("GT") == L else ""
    print(f"{L}. {rec[L]}{marker}")
print(f"\ngold chunk ids ({len(gold_ids)}): {sorted(gold_ids)}")

## Step 2 — the perception (stand-in for a real PERCEIVE call)

`observed` matters for routing (§1: sign/marking mentions come from the
image, not always the question). `violation` is what actually gets
retrieved with (§2) — statute-register phrasing, same discipline as the
shipped pipeline's PERCEIVE step.

In [ ]:
MY_PERCEPTION = Perception(
    focus="REPLACE ME",
    vehicle="xe mô tô",
    observed="REPLACE ME — scene language, what's actually visible (signs, lights, lane markings...)",
    violation="REPLACE ME — statute-register violation phrase",
    confidence=0.8,
)

print(MY_PERCEPTION)

## Step 3 — routing decision (§1)

Runs on `question + perception.observed`. Shows which of `{LUAT, ND, QCVN}`
fired and why — compare against routing on the question text ALONE to see
what perception adds (this is exactly the item-129-style case QUERY_GEN_SPEC.md
§1 is built around: a generic question with no routing signal at all, where
only the image tells you it needs QCVN).

In [ ]:
routed_text_only = route(rec["question"])
routed_with_perception = route(rec["question"], MY_PERCEPTION)

print(f"routed (question text alone): {routed_text_only}")
print(f"routed (question + perception.observed): {routed_with_perception}")
if routed_with_perception != routed_text_only:
    print(f"  -> perception changed the routing decision: added {routed_with_perception - routed_text_only}")

## Step 4 — retrieve within the routed documents (§2)

Budget `K_TOTAL` split EVENLY across whatever routed — not a global top-k,
and zero budget for anything not routed. This is the base retrieval, BEFORE
citation-graph expansion.

In [ ]:
base_retrieved = retrieve_within_routed(
    MY_PERCEPTION.violation, routed_with_perception, K_TOTAL, corpus, retriever, AS_OF
)

by_doc = {}
for c in base_retrieved:
    by_doc.setdefault(corpus.by_id[c.chunk_id].document_id, []).append(c.chunk_id)
for doc, ids in by_doc.items():
    print(f"{doc} ({len(ids)}): {ids}")

base_overlap = {c.chunk_id for c in base_retrieved} & gold_ids
print(f"\nbase retrieval overlap with gold: {sorted(base_overlap)}")

## Step 5 — close the citation graph (§3)

1 hop, additive — appended AFTER the base retrieval, never displacing it.
`graph_rev`: every neighbor of a retrieved chunk that points AT it (capped
`MAX_REV`, no reranking needed). `graph_fwd`: every chunk a retrieved chunk
points AT, reranked by BM25 against `MY_PERCEPTION.violation` and capped
`MAX_FWD` — a sanction chunk's fan-out covers multiple conduct provisions,
so this is what picks out the right one.

In [ ]:
tagged = expand_with_graph(base_retrieved, graph, MY_PERCEPTION.violation, corpus, MAX_REV, MAX_FWD)

n_by_source = {}
for t in tagged:
    n_by_source[t.source] = n_by_source.get(t.source, 0) + 1
print(f"chunks by source: {n_by_source}\n")

for t in tagged:
    marker = "  <- GOLD" if t.chunk.chunk_id in gold_ids else ""
    print(f"[{t.source:9}] {t.chunk.chunk_id}{marker}")

## Step 6 — retrieved (all sources) vs. gold

In [ ]:
all_ids = [t.chunk.chunk_id for t in tagged]
overlap = set(all_ids) & gold_ids
missing = gold_ids - set(all_ids)

print(f"retrieved ({len(all_ids)}): {all_ids}")
print(f"gold      ({len(gold_ids)}): {sorted(gold_ids)}")
print(f"overlap   ({len(overlap)}): {sorted(overlap)}")
if missing:
    print(f"MISSED    ({len(missing)}): {sorted(missing)}")
    recovered_by_expand = missing & {t.chunk.chunk_id for t in tagged if t.source != "retrieval"}
    still_missing = missing - recovered_by_expand
    if recovered_by_expand:
        print(f"  (of which recovered by graph expansion, not base retrieval: {sorted(recovered_by_expand)})")
    if still_missing:
        print(f"  (still missed after expansion: {sorted(still_missing)})")

## Step 7 (optional) — read the full chunk text

Useful for judging whether a retrieved-but-not-gold chunk is a reasonable
near-miss, or whether a missed gold chunk would need a 2nd hop (not
implemented — QUERY_GEN_SPEC.md §3/§5 gate: "2 hops explodes the pool").

In [ ]:
print("RETRIEVED chunks:\n")
for t in tagged:
    marker = "  <- GOLD" if t.chunk.chunk_id in gold_ids else ""
    print(f"[{t.source}] [{t.chunk.chunk_id}] score={t.chunk.score:.4f}{marker}")
    print(f"{t.chunk.content}\n")

if missing:
    print("MISSED gold chunks (in gold, never retrieved even after expansion):\n")
    for cid in sorted(missing):
        chunk = corpus.by_id.get(cid)
        text = chunk.text if chunk else "(not in corpus!)"
        print(f"[{cid}]")
        print(f"{text}\n")